In [ ]:
# data sets
# from torch_geometric.datasets import QM9 # we will not be using QM9
from tdc.utils import get_reaction_type

# rdkit things
from rdkit import Chem
from rdkit.Chem import GetPeriodicTable
from rdkit.Chem import rdChemReactions
from rdkit.Chem import Draw
from rdkit.Chem.Draw import ReactionToImage
from rdkit.Chem.Draw import MolToImage

# standard libraries
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# machine learning libraries
import torch
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split


### (1) Retrieve the cleaned data from data_exploration.ipynb to use in our model.

In [3]:
alchemy_elements = {'C', 'Cl', 'F', 'H', 'N', 'O', 'S'}
molecule_types_df = pd.read_csv("Collected Data/Alchemy-v20191129/final_version_with_smiles.csv")
reaction_types_df = pd.read_csv("Collected Data/uspto_50k_processed.csv")

display(molecule_types_df)
display(reaction_types_df)

,gdb_idx,atom number,"zpve\n(Ha, zero point vibrational energy)","Cv\n(cal/molK, heat capacity at 298.15 K)","gap\n(Ha, LUMO-HOMO)","G\n(Ha, Free energy at 298.15 K)","HOMO\n(Ha, energy of HOMO)","U\n(Ha, internal energy at 298.15 K)","alpha\n(a_0^3, Isotropic polarizability)","U0\n(Ha, internal energy at 0 K)","H\n(Ha, enthalpy at 298.15 K)","LUMO\n(Ha, energy of LUMO)","mu\n(D, dipole moment)","R2\n(a_0^2, electronic spatial extent)",smiles
0,2859833,9,0.226164,0.000067,0.300531,-352.128828,-0.236722,-352.076085,99.201621,-352.088194,-352.075141,0.063809,0.180558,1919.249225,CC#CC[C@H](C)CCC
1,3148292,9,0.180782,0.000058,0.293098,-386.820766,-0.241536,-386.774917,84.670738,-386.784878,-386.773973,0.051562,1.302656,1420.242859,C#CC[C@@H]1CO[C@@H]1CC
2,3607838,9,0.205922,0.000057,0.307544,-388.062328,-0.224965,-388.018758,86.198178,-388.028208,-388.017814,0.082579,1.284923,1257.238492,CC[C@H]1[C@H]2CCO[C@]21C
3,9540153,11,0.160845,0.000056,0.195923,-515.046508,-0.203538,-515.001198,95.801267,-515.010568,-515.000254,-0.007615,4.074910,1600.220066,COc1nccc2c1OCC2
4,340363,10,0.177712,0.000057,0.254495,-403.843164,-0.248943,-403.800346,95.147725,-403.809426,-403.799402,0.005552,4.111533,1442.171110,CC1=CC[C@H]2C[C@@H]2[C@H]1C#N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202574,9581016,10,0.120222,0.000051,0.179768,-450.915828,-0.244325,-450.872449,88.581216,-450.881390,-450.871505,-0.064557,6.434535,1541.115465,CNc1cc(C#N)ncn1
202575,9706527,10,0.130179,0.000053,0.189401,-438.632624,-0.231340,-438.589755,96.102362,-438.598674,-438.588811,-0.041938,0.338522,1628.839361,C#Cc1ccnc(OC)c1
202576,9760179,10,0.119919,0.000051,0.163827,-450.920199,-0.232783,-450.875248,89.074794,-450.884399,-450.874304,-0.068956,4.337358,1497.024559,CNc1nccc(C#N)n1
202577,9845175,10,0.119740,0.000050,0.191354,-454.722866,-0.251374,-454.680504,89.026059,-454.689065,-454.679560,-0.060020,5.263580,1611.219983,COc1cncc(C#N)c1


,reactant,product,category
0,C1=COCCC1.COC(=O)CCC(=O)c1ccc(O)cc1O,COC(=O)CCC(=O)c1ccc(OC2CCCCO2)cc1O,1
1,COC(=O)c1cccc(C(=O)O)c1.Nc1cccnc1N,COC(=O)c1cccc(-c2nc3cccnc3[nH]2)c1,4
2,CC(C)(C)OC(=O)NC1CCC(C(=O)O)CC1.CNOC,CON(C)C(=O)C1CCC(NC(=O)OC(C)(C)C)CC1,2
3,Nc1ccc(O)cc1.O=[N+]([O-])c1ccc(Cl)nc1Cl,O=[N+]([O-])c1ccc(Cl)nc1Nc1ccc(O)cc1,1
4,[N-]=[N+]=NCC1=CC[C@@H](c2ccc(Cl)cc2Cl)[C@H]([...,NCC1=CC[C@@H](c2ccc(Cl)cc2Cl)[C@H]([N+](=O)[O-...,9
...,...,...,...
35111,COC(=O)c1ccc(C(=O)OCc2ccccc2)cc1F,COC(=O)c1ccc(C(=O)O)cc1F,6
35112,C=CCOC(=O)N1CCc2nnc(NN)cc2C1.CC(C)=O,C=CCOC(=O)N1CCc2nnc(NN=C(C)C)cc2C1,1
35113,FC1(F)CCNC1.O=C(O)c1ccncc1NC(=O)c1nc(C2CC2)ccc...,O=C(Nc1cnccc1C(=O)N1CCC(F)(F)C1)c1nc(C2CC2)ccc...,2
35114,CC(=O)Cl.CC(C)(C)OC(=O)NCCO,CC(=O)OCCNC(=O)OC(C)(C)C,2


### (2) Focus on making the GNN with the alchemy DFT data. Use only a fraction of the data so I can practically run this.

In [6]:
# Create a smaller dataframe with a random sample of molecules from the alchemy dataset to work with locally. 

n_examples = 2000
small_molecule_types_df = molecule_types_df.sample(n=n_examples, random_state=42).reset_index(drop=True)
display(small_molecule_types_df)

,gdb_idx,atom number,"zpve\n(Ha, zero point vibrational energy)","Cv\n(cal/molK, heat capacity at 298.15 K)","gap\n(Ha, LUMO-HOMO)","G\n(Ha, Free energy at 298.15 K)","HOMO\n(Ha, energy of HOMO)","U\n(Ha, internal energy at 298.15 K)","alpha\n(a_0^3, Isotropic polarizability)","U0\n(Ha, internal energy at 0 K)","H\n(Ha, enthalpy at 298.15 K)","LUMO\n(Ha, energy of LUMO)","mu\n(D, dipole moment)","R2\n(a_0^2, electronic spatial extent)",smiles
0,1365523,10,0.176608,0.000059,0.234020,-478.159623,-0.229535,-478.113309,86.159364,-478.123454,-478.112364,0.004485,0.304657,1622.044495,Cc1ncc(CCCO)o1
1,3349991,10,0.196003,0.000069,0.206017,-442.119354,-0.235361,-442.069665,97.414650,-442.081465,-442.068721,-0.029343,0.458808,1787.349824,C#CC(CCC(C)C)=NO
2,1913236,10,0.258195,0.000070,0.317400,-428.532395,-0.255799,-428.483912,99.397638,-428.495522,-428.482968,0.061602,1.703904,1922.446278,CC[C@H]1CC[C@@H](CO)[C@H]1C
3,7381920,11,0.206574,0.000065,0.242678,-517.407334,-0.229677,-517.359413,95.398618,-517.370302,-517.358468,0.013000,2.968662,1860.400373,O=C1CC=CCCN1CCO
4,5665800,10,0.148714,0.000058,0.311669,-836.975181,-0.291001,-836.928780,85.926614,-836.938784,-836.927836,0.020668,2.153929,2032.372758,N#CCCC1CS(=O)(=O)C1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,6215982,10,0.200927,0.000061,0.243524,-442.225015,-0.227106,-442.180515,92.191559,-442.190422,-442.179571,0.016418,3.770300,1417.383020,C[C@@H]1CC=CCC(=O)N1C
1996,326171,11,0.183276,0.000059,0.191066,-479.121834,-0.201582,-479.074783,105.261641,-479.084830,-479.073838,-0.010516,0.995386,2188.051116,Cc1nc(C2=CCCC2)co1
1997,411714,11,0.191727,0.000067,0.267669,-574.332093,-0.216469,-574.284666,84.853033,-574.295783,-574.283722,0.051201,0.902507,1758.705259,C[C@]12O[C@@H](CO)CO[C@H]1[C@H]2O
1998,8998250,10,0.141128,0.000056,0.188286,-455.882830,-0.255220,-455.838132,83.981487,-455.847704,-455.837188,-0.066934,3.572779,1419.589445,CN[C@@]1(C#N)C=CC(=O)C1


In [ ]:
def smiles_to_graph(smi): 
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    
    # atom features
    atom_features = [atom.GetAtomicNum() for atom in mol.GetAtoms()]
    
    # edges
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append((i, j))
        edge_index.append((j, i))
    
    # convert to tensor
    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    
    x = torch.tensor(atom_features, dtype=torch.float).unsqueeze(1)
    
    return Data(x=x, edge_index=edge_index)

